In [1]:
import sys
import time
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

root_dir = Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from langchain_groq import ChatGroq
from src.rag_pipeline import (
    load_hybrid_retriever,
    relevant_text_hybrid,
    clean_response
)
from src.prompts import build_prompt, SYSTEM_PROMPT

In [2]:
# load processed data
data_path = root_dir / "data" / "processed" / "processed.parquet"
df = pd.read_parquet(data_path)
df_by_asin = df.set_index("parent_asin", drop=False)
print(f"Loaded {len(df)} products")

# load hybrid retriever, both LLM models to use same retrieved results
hybrid_retriever = load_hybrid_retriever()
print("Hybrid retriever loaded successfully")

Loaded 20000 products


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hybrid retriever loaded successfully


In [3]:
# milestone 2 model (baseline): qwen/qwen3-32b
llm_qwen = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0.2,
    max_tokens=3000,
    )

# milestone 3 model (experiment): meta-llama/llama-4-scout-17b-16e-instruct
llm_llama = ChatGroq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    temperature=0.2, # same creativity level as qwen3-32b
    max_tokens=3000, # same max tokens as qwen3-32b
    )

In [4]:
# define the 5 evaluation queries for the experiment
eval_queries = [
    "Washing machine",
    "Stainless steel coffee maker",
    "Magic Bullet",
    "kitchen device to heat food quickly",
    "Best appliances for a small apartment",
]

In [5]:
results = []

for query in eval_queries:
    print(f"Running query: '{query}'")

    retrieved_docs = hybrid_retriever.retrieve(query, top_k=5) # retrieve top 5 relevant docs
    context = relevant_text_hybrid(retrieved_docs, df_by_asin) # build context from retrieved docs
    prompt = build_prompt(query, context) # build prompt

    # response from both models using same prompt and context
    response_qwen = clean_response(llm_qwen.invoke(prompt).content)
    response_llama = clean_response(llm_llama.invoke(prompt).content)

    results.append({
        "query": query,
        "context": context,
        "prompt": prompt,
        "qwen3_32b": response_qwen,
        "llama4_scout": response_llama,
    })

    print(f"Done.")
    time.sleep(15) # add delay to avoid hitting rate limits

print("All queries complete.")

Running query: 'Washing machine'
Done.
Running query: 'Stainless steel coffee maker'
Done.
Running query: 'Magic Bullet'
Done.
Running query: 'kitchen device to heat food quickly'
Done.
Running query: 'Best appliances for a small apartment'
Done.
All queries complete.


In [6]:
for result in results:
    print("=" * 79)
    print(f"Query: {result['query']}")
    print("=" * 79)
    print(f"qwen3-32b:\n{result['qwen3_32b']}")
    print("=" * 79)
    print(f"llama-4-scout:\n{result['llama4_scout']}")
    print()

Query: Washing machine
qwen3-32b:
Based on the Amazon datasets, here are two notable washing machine options:  

1. **Formemory Portable Folding Bucket Turbo Ultrasonic Washing Machine** (ASIN: B08FMND788):  
   - **Rating**: 5.0 (1 review, 2 helpful votes).  
   - **Key features**: Compact, portable, and folds for storage. Ideal for small/light loads (e.g., pet clothing). Includes remote control.  

2. **Midea 1.6 CF Portable Washing Machine** (ASIN: B00DJF6296):  
   - **Rating**: 4.5 (8 reviews, 4 helpful votes).  
   - **Key features**: Handles larger loads (e.g., queen-sized sheets, jeans). Users note it requires a 12-foot water hose extension for drainage.  

For a budget-friendly option with no reviews yet, consider the **Pink Mini USB Portable Washing Machine** (ASIN: B09L1LHFP8, $17.89).
llama-4-scout:
If you're looking for a compact washing machine, consider the Midea 1.6 CF Portable Washing Machine Washer (ASIN: B00DJF6296) with a 4.5-star rating. It's a great option for han

In [7]:
print("System prompt used for all queries in both models:")
print(SYSTEM_PROMPT)

System prompt used for all queries in both models:

    You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible.
    Be concise and specific.
